In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "workspace.desafio_sicredi"

print("Ambiente:", BASE)

Ambiente: workspace.desafio_sicredi


In [0]:
df_origem = spark.table(
    f"{BASE}.silver_transacoes_cartao"
)

print("Transações na origem:", df_origem.count())

display(
    df_origem.select(
        "id_transacao",
        "id_cartao",
        "valor",
        "valor_liquido",
        "estornada"
    )
)

Transações na origem: 5


id_transacao,id_cartao,valor,valor_liquido,estornada
5001,1005,150.00,150.00,false
5002,1005,80.00,0.00,true
5004,1003,300.00,300.00,false
5006,1005,200.00,200.00,false
5008,1005,450.00,450.00,false


In [0]:
TABELA_DESTINO = f"{BASE}.silver_transacoes_incremental_teste"

(
    df_origem.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_DESTINO)
)

print("Tabela criada:", TABELA_DESTINO)

Tabela criada: workspace.desafio_sicredi.silver_transacoes_incremental_teste


In [0]:
from delta.tables import DeltaTable

delta_destino = DeltaTable.forName(
    spark,
    TABELA_DESTINO
)

(
    delta_destino.alias("destino")
    .merge(
        df_origem.alias("origem"),
        "destino.id_transacao = origem.id_transacao"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE executado com sucesso!")

MERGE executado com sucesso!


In [0]:
df_resultado = spark.table(TABELA_DESTINO)

print("Total após MERGE:", df_resultado.count())

print(
    "IDs distintos:",
    df_resultado.select("id_transacao").distinct().count()
)

Total após MERGE: 5
IDs distintos: 5


In [0]:
from pyspark.sql import Row

df_nova_transacao = (
    df_origem
    .filter(F.col("id_transacao") == 5001)
    .withColumn(
        "id_transacao",
        F.lit(6001).cast("bigint")
    )
    .withColumn(
        "valor",
        F.lit(250).cast("decimal(18,2)")
    )
    .withColumn(
        "valor_liquido",
        F.lit(250).cast("decimal(18,2)")
    )
    .withColumn(
        "estornada",
        F.lit(False)
    )
)

display(
    df_nova_transacao.select(
        "id_transacao",
        "valor",
        "valor_liquido"
    )
)

id_transacao,valor,valor_liquido
6001,250.00,250.00


In [0]:
(
    delta_destino.alias("destino")
    .merge(
        df_nova_transacao.alias("origem"),
        "destino.id_transacao = origem.id_transacao"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Nova transação processada!")

Nova transação processada!


In [0]:
df_final = spark.table(TABELA_DESTINO)

print("Total de transações:", df_final.count())

display(
    df_final.select(
        "id_transacao",
        "valor",
        "valor_liquido"
    ).orderBy("id_transacao")
)

Total de transações: 6


id_transacao,valor,valor_liquido
5001,150.00,150.00
5002,80.00,0.00
5004,300.00,300.00
5006,200.00,200.00
5008,450.00,450.00
6001,250.00,250.00


In [0]:
from pyspark.sql import functions as F

BASE = "workspace.desafio_sicredi"
TABELA_DESTINO = f"{BASE}.silver_transacoes_incremental_teste"

df_atualizacao = (
    spark.table(TABELA_DESTINO)
    .filter(F.col("id_transacao") == 6001)
    .withColumn(
        "valor",
        F.lit(275).cast("decimal(18,2)")
    )
    .withColumn(
        "valor_liquido",
        F.lit(275).cast("decimal(18,2)")
    )
)

display(
    df_atualizacao.select(
        "id_transacao",
        "valor",
        "valor_liquido"
    )
)

id_transacao,valor,valor_liquido
6001,275.00,275.00


In [0]:
from delta.tables import DeltaTable

delta_destino = DeltaTable.forName(
    spark,
    TABELA_DESTINO
)

(
    delta_destino.alias("destino")
    .merge(
        df_atualizacao.alias("origem"),
        "destino.id_transacao = origem.id_transacao"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Atualização incremental realizada!")

Atualização incremental realizada!


In [0]:
df_validacao = spark.table(TABELA_DESTINO)

display(
    df_validacao
    .filter(F.col("id_transacao") == 6001)
    .select(
        "id_transacao",
        "valor",
        "valor_liquido"
    )
)

print("Total de registros:", df_validacao.count())

print(
    "IDs distintos:",
    df_validacao.select("id_transacao").distinct().count()
)

id_transacao,valor,valor_liquido
6001,275.00,275.00


Total de registros: 6
IDs distintos: 6


In [0]:
from pyspark.sql import functions as F

BASE = "workspace.desafio_sicredi"

dados_setembro = [
    (1, "Ana Maria Silva", "2026-09-01", 400.00),
    (3, "Mariana Costa", "2026-09-01", 250.00)
]

df_setembro = spark.createDataFrame(
    dados_setembro,
    ["id_cliente", "nome_cliente", "mes_referencia", "valor_liquido"]
).withColumn(
    "mes_referencia",
    F.to_date("mes_referencia")
).withColumn(
    "valor_liquido",
    F.col("valor_liquido").cast("decimal(18,2)")
)

display(df_setembro)


id_cliente,nome_cliente,mes_referencia,valor_liquido
1,Ana Maria Silva,2026-09-01,400.00
3,Mariana Costa,2026-09-01,250.00


In [0]:
df_outubro = (
    spark.table(f"{BASE}.gold_resumo_cliente_mes")
    .select(
        "id_cliente",
        "nome_cliente",
        "mes_referencia",
        "valor_liquido"
    )
)

display(df_outubro)

id_cliente,nome_cliente,mes_referencia,valor_liquido
1,Ana Maria Silva,2026-10-01T00:00:00.000Z,800.00
3,Mariana Costa,2026-10-01T00:00:00.000Z,300.00


In [0]:
df_evolucao = df_setembro.unionByName(df_outubro)

(
    df_evolucao.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BASE}.teste_evolucao_mensal")
)

display(df_evolucao.orderBy("id_cliente", "mes_referencia"))

id_cliente,nome_cliente,mes_referencia,valor_liquido
1,Ana Maria Silva,2026-09-01T00:00:00.000Z,400.00
1,Ana Maria Silva,2026-10-01T00:00:00.000Z,800.00
3,Mariana Costa,2026-09-01T00:00:00.000Z,250.00
3,Mariana Costa,2026-10-01T00:00:00.000Z,300.00


In [0]:
%sql

WITH evolucao AS (

    SELECT
        id_cliente,
        nome_cliente,
        mes_referencia,
        valor_liquido,

        LAG(valor_liquido) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS valor_mes_anterior,

        LAG(mes_referencia) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS mes_anterior

    FROM workspace.desafio_sicredi.teste_evolucao_mensal
)

SELECT
    nome_cliente,
    mes_referencia,
    valor_mes_anterior,
    valor_liquido,

    ROUND(
        (
            (valor_liquido - valor_mes_anterior)
            / NULLIF(valor_mes_anterior, 0)
        ) * 100,
        2
    ) AS crescimento_percentual

FROM evolucao

WHERE mes_anterior = ADD_MONTHS(mes_referencia, -1)

ORDER BY crescimento_percentual DESC;

nome_cliente,mes_referencia,valor_mes_anterior,valor_liquido,crescimento_percentual
Ana Maria Silva,2026-10-01T00:00:00.000Z,400.00,800.00,100.00
Mariana Costa,2026-10-01T00:00:00.000Z,250.00,300.00,20.00


In [0]:
%sql

WITH evolucao AS (

    SELECT
        id_cliente,
        nome_cliente,
        mes_referencia,
        valor_liquido,

        LAG(valor_liquido) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS valor_mes_anterior,

        LAG(mes_referencia) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS mes_anterior

    FROM workspace.desafio_sicredi.teste_evolucao_mensal
)

SELECT
    nome_cliente,
    valor_mes_anterior,
    valor_liquido,

    ROUND(
        (
            (valor_liquido - valor_mes_anterior)
            / NULLIF(valor_mes_anterior, 0)
        ) * 100,
        2
    ) AS crescimento_percentual

FROM evolucao

WHERE mes_anterior = ADD_MONTHS(mes_referencia, -1)
  AND valor_mes_anterior > 0
  AND valor_liquido > valor_mes_anterior * 1.5;

nome_cliente,valor_mes_anterior,valor_liquido,crescimento_percentual
Ana Maria Silva,400.00,800.00,100.00


In [0]:
%sql

WITH movimentacao_cliente AS (

    SELECT
        id_cliente,
        nome_cliente,
        SUM(valor_liquido) AS valor_total

    FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

    GROUP BY
        id_cliente,
        nome_cliente
)

SELECT
    id_cliente,
    nome_cliente,
    valor_total,

    NTILE(4) OVER (
        ORDER BY valor_total DESC, id_cliente
    ) AS quartil_movimentacao

FROM movimentacao_cliente

ORDER BY quartil_movimentacao;

id_cliente,nome_cliente,valor_total,quartil_movimentacao
1,Ana Maria Silva,800.00,1
3,Mariana Costa,300.00,2


In [0]:
%sql

WITH movimentacao_cliente AS (

    SELECT
        id_cliente,
        nome_cliente,
        SUM(valor_liquido) AS valor_total

    FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

    GROUP BY id_cliente, nome_cliente

),

classificacao AS (

    SELECT
        *,
        NTILE(4) OVER (
            ORDER BY valor_total DESC, id_cliente
        ) AS quartil

    FROM movimentacao_cliente
)

SELECT
    quartil,
    COUNT(*) AS quantidade_clientes,
    SUM(valor_total) AS valor_movimentado

FROM classificacao

GROUP BY quartil

ORDER BY quartil;

quartil,quantidade_clientes,valor_movimentado
1,1,800.00
2,1,300.00
